# 05 — LSTM Model

Train an LSTM using the same fixed train/test split as Logistic Regression. The tokenizer is fitted only on training text.

In [ ]:
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
ROOT = Path.cwd(); DATA = ROOT / "data" / "processed" / "cleaned_news.csv"; SPLIT = ROOT / "data" / "processed" / "model_split.csv"
MODELS = ROOT / "models"; METRICS = ROOT / "reports" / "metrics"; FIGURES = ROOT / "reports" / "figures"
MODELS.mkdir(exist_ok=True); METRICS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA); split = pd.read_csv(SPLIT)
train_idx = split.loc[split.split == "train", "row_index"].to_numpy(); test_idx = split.loc[split.split == "test", "row_index"].to_numpy()
X_train, y_train = df.loc[train_idx, "clean_text"].astype(str), df.loc[train_idx, "label"].astype(int)
X_test, y_test = df.loc[test_idx, "clean_text"].astype(str), df.loc[test_idx, "label"].astype(int)

In [ ]:
MAX_WORDS = 20000
MAX_LEN = 300
OOV_TOKEN = "<OOV>"
EMBEDDING_DIM = 128

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train)
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding="post", truncating="post")

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
history = model.fit(X_train_pad, y_train, epochs=8, batch_size=64, validation_split=0.20, callbacks=[early_stopping], verbose=1)

In [ ]:
y_prob = model.predict(X_test_pad, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
metrics = {
    "model": "LSTM",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "test_size": len(y_test),
    "max_words": MAX_WORDS,
    "max_len": MAX_LEN,
    "embedding_dim": EMBEDDING_DIM,
}
print(metrics)
print(classification_report(y_test, y_pred, target_names=["Fake News", "Real News"], zero_division=0))

In [ ]:
model.save(MODELS / "lstm_model.keras")
with open(MODELS / "lstm_tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer.to_json())
pd.DataFrame([metrics]).to_csv(METRICS / "lstm_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5)); ax.plot(history.history["accuracy"], label="Train"); ax.plot(history.history["val_accuracy"], label="Validation"); ax.set_title("LSTM Accuracy"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / "lstm_accuracy.png", dpi=150); plt.show()
fig, ax = plt.subplots(figsize=(8, 5)); ax.plot(history.history["loss"], label="Train"); ax.plot(history.history["val_loss"], label="Validation"); ax.set_title("LSTM Loss"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES / "lstm_loss.png", dpi=150); plt.show()

cm = confusion_matrix(y_test, y_pred); fig, ax = plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm, display_labels=["Fake", "Real"]).plot(ax=ax); ax.set_title("LSTM Confusion Matrix"); fig.tight_layout(); fig.savefig(FIGURES / "lstm_confusion_matrix.png", dpi=150); plt.show()

In [ ]:
def predict_news_lstm(text: str):
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding="post", truncating="post")
    probability = float(model.predict(padded, verbose=0)[0, 0])
    return ("Real News" if probability >= 0.5 else "Fake News", probability)

predict_news_lstm("Government officials announced new economic reforms after a press conference.")